In [39]:
import numpy as np

# ======================================================
# Configuration (edit to match RTL)
# ======================================================
Fs_in   = 6_000_000     # input sample rate (Hz)
R       = 8             # decimation factor (1,2,4,8,16)
N_samps = 16384         # stimulus length (used if you generate test input here)

# I/O and coeff formats
# - Input/output samples: s16.15 (stored as int16, arithmetic in int32)
# - FIR taps: s1.19 (stored as int32 Q19)
Q_IN_FRAC   = 15
Q_COEF_FRAC = 19
Q_PROD_FRAC = Q_IN_FRAC + Q_COEF_FRAC  # 34

INT16_MIN, INT16_MAX = -32768, 32767

stim_file = "cic_stimulus.txt"   # input samples for RTL
exp_file  = "cic_expected.txt"   # expected compensated outputs (s16.15)

# ======================================================
# Fixed-point helpers (hardware-realistic)
# ======================================================
def sat_to_width(x: int, bits: int) -> int:
    mask = (1 << bits) - 1
    y = x & mask
    return y - (1 << bits) if (y & (1 << (bits - 1))) else y

def sign_extend(x: int, from_bits: int, to_bits: int) -> int:
    y = sat_to_width(x, from_bits)
    if y & (1 << (from_bits - 1)):
        y |= (~((1 << from_bits) - 1))
    else:
        y &= ((1 << from_bits) - 1)
    return sat_to_width(y, to_bits)

def round_shift_right(x: int, shift: int) -> int:
    # Round-to-nearest, ties to +inf (add 0.5 LSB before arithmetic shift)
    if shift <= 0:
        return x
    bias = 1 << (shift - 1)
    return (x + bias) >> shift

def sat16(x: int) -> int:
    return INT16_MAX if x > INT16_MAX else (INT16_MIN if x < INT16_MIN else x)

def sanitize_R(R):
    return R if R in (1,2,4,8,16) else 1

# ======================================================
# CIC decimator core (4-stage, M=1), RTL-accurate truncation, no normalization
# Integrators width: 21, comb subtract width: 21, stage truncation to 20 bits
# Output widened to 32 bits for downstream FIR
# ======================================================
def cic_core_s16_to_s32(input_q15: np.ndarray, R: int) -> np.ndarray:
    R = sanitize_R(R)
    i1=i2=i3=i4=0
    d1=d2=d3=d4=0
    cnt=0
    outputs=[]
    for x_raw in input_q15:
        # s16.15 sample -> use integer carrier, extend to 21 for accumulation
        x21 = sign_extend(int(x_raw), 16, 21)

        # 4 integrators with stage truncation to 20 bits (guarded by 21-bit acc)
        i1 = sat_to_width(i1 + x21, 21); i1_20 = sat_to_width(i1, 20)
        i2 = sat_to_width(i2 + sign_extend(i1_20, 20, 21), 21); i2_20 = sat_to_width(i2, 20)
        i3 = sat_to_width(i3 + sign_extend(i2_20, 20, 21), 21); i3_20 = sat_to_width(i3, 20)
        i4 = sat_to_width(i4 + sign_extend(i3_20, 20, 21), 21); i4_20 = sat_to_width(i4, 20)

        # Decimate by R: combs run only when cnt==0
        if cnt == 0:
            ds = i4_20
            # 4 combs, subtract in 21 bits, truncate to 20 after each stage
            sub1 = sat_to_width(sign_extend(ds, 20, 21) - sign_extend(d1, 20, 21), 21)
            y1   = sat_to_width(sub1, 20); d1 = ds

            sub2 = sat_to_width(sign_extend(y1, 20, 21) - sign_extend(d2, 20, 21), 21)
            y2   = sat_to_width(sub2, 20); d2 = y1

            sub3 = sat_to_width(sign_extend(y2, 20, 21) - sign_extend(d3, 20, 21), 21)
            y3   = sat_to_width(sub3, 20); d3 = y2

            sub4 = sat_to_width(sign_extend(y3, 20, 21) - sign_extend(d4, 20, 21), 21)
            y4   = sat_to_width(sub4, 20); d4 = y3

            # widen to 32 bits for FIR MAC input
            outputs.append(sign_extend(y4, 20, 32))

        cnt = cnt + 1 if cnt < (R - 1) else 0

    return np.array(outputs, dtype=np.int32)

# ======================================================
# FIR compensation (fixed-point, hardware-realistic)
# - taps_q19: int32 array of s1.19 coefficients
# - input: s32 (CIC output integer carrier)
# - product: s17.34, accumulate in wide int64
# - output: rounded and saturated to s16.15
# ======================================================
def fir_compensate_s32_to_s16(cic_out_s32: np.ndarray, taps_q19: np.ndarray) -> np.ndarray:
    ntaps = len(taps_q19)
    # Shift register buffer (int32)
    buf = np.zeros(ntaps, dtype=np.int32)
    y_out = np.zeros_like(cic_out_s32, dtype=np.int16)

    for n, x in enumerate(cic_out_s32):
        # shift in new sample
        buf[1:] = buf[:-1]
        buf[0] = int(x)

        # folded MAC if symmetric taps could be added; here full MAC for general taps
        acc = 0  # int64 accumulator
        for k in range(ntaps):
            # product in s17.34: buf (s16.15 carrier in int32) * tap (s1.19)
            # treat buf as s16.15 integer; multiply by s1.19 → s17.34 integer
            prod = int(buf[k]) * int(taps_q19[k])  # int64
            acc += prod

        # Round from Q34 down to Q15 (difference = 19)
        y_q15 = round_shift_right(acc, Q_COEF_FRAC)  # shift by 19

        # Saturate to int16 range
        y_out[n] = sat16(y_q15)

    return y_out

# ======================================================
# Tap quantization helper (float -> s1.19)
# ======================================================
def quantize_taps_s1_19(taps_float: np.ndarray) -> np.ndarray:
    scale = 1 << Q_COEF_FRAC
    q = np.round(taps_float * scale).astype(np.int64)
    # Clamp to s1.19 range
    q_min = -(1 << Q_COEF_FRAC)
    q_max =  (1 << Q_COEF_FRAC) - 1
    q = np.clip(q, q_min, q_max).astype(np.int32)
    return q

# ======================================================
# Stimulus generation (optional, for local testing)
# ======================================================
def generate_stimulus(Fs: int, N: int) -> np.ndarray:
    t = np.arange(N) / Fs
    x_float = 0.6*np.sin(2*np.pi*200e3*t) + 0.2*np.sin(2*np.pi*900e3*t)
    x_q15   = np.clip(np.round(x_float * (1 << Q_IN_FRAC)), INT16_MIN, INT16_MAX).astype(np.int16)
    return x_q15

# ======================================================
# Entry point: read or generate inputs, run CIC + FIR, write expected
# Replace taps_float with your real compensation FIR taps (float), or provide s1.19 directly
# ======================================================
if __name__ == "__main__":
    # 1) Prepare input (generate or read)
    x_q15 = generate_stimulus(Fs_in, N_samps)

    # 2) CIC core (fixed-point only)
    y_cic_s32 = cic_core_s16_to_s32(x_q15, R)

    # 3) FIR taps: provide your real compensation taps here (float), then quantize to s1.19
    # Placeholder: unity gain single-tap (no compensation). Replace with real taps.
    taps_float = np.array([1.0], dtype=np.float64)

    # Quantize taps to s1.19
    taps_q19 = quantize_taps_s1_19(taps_float)

    # 4) FIR compensate (fixed-point MAC, round-to-nearest, saturate)
    y_out_q15 = fir_compensate_s32_to_s16(y_cic_s32, taps_q19)

    # 5) Write stimulus and expected outputs
    with open(stim_file, "w") as f:
        for v in x_q15:
            f.write(f"{int(v)}\n")

    with open(exp_file, "w") as f:
        for v in y_out_q15:
            f.write(f"{int(v)}\n")

    # 6) Minimal report
    print(f"[SUCCESS] Stimulus written to {stim_file}, Expected outputs written to {exp_file}")
    print(f"[INFO] R={R}, CIC_out={len(y_cic_s32)}, FIR_taps={len(taps_q19)}")
    print(f"[INFO] Output range: {y_out_q15.min()}..{y_out_q15.max()}")
    from numpy.fft import rfft, rfftfreq

def analyze_chain_response(R, Fs_in, taps_q19, pb_edge=0.40, sb_edge=0.48, Nimp=16384):
    """
    Compute passband ripple and stopband attenuation of CIC+FIR chain.
    - R: decimation factor
    - Fs_in: input sample rate (Hz)
    - taps_q19: FIR taps (s1.19 int32)
    - pb_edge: passband edge (normalized to Nyquist, 0..1)
    - sb_edge: stopband start (normalized to Nyquist, 0..1)
    - Nimp: impulse length
    Returns: (ripple_dB, stopband_att_dB)
    """
    # Build impulse input
    x = np.zeros(Nimp, dtype=np.int16)
    x[0] = 32767  # full-scale impulse

    # Run CIC
    y_cic = cic_core_s16_to_s32(x, R)

    # Run FIR compensation
    y_eq  = fir_compensate_s32_to_s16(y_cic, taps_q19)

    # FFT analysis
    Fs_out = Fs_in / R
    win = np.hanning(len(y_eq))
    spec = np.abs(rfft(y_eq.astype(np.float64) * win))
    spec_db = 20*np.log10(spec / (np.max(spec) + 1e-12))

    freqs = rfftfreq(len(y_eq), d=1/Fs_out)
    f_norm = freqs / (Fs_out/2)  # normalized to Nyquist

    # Passband ripple
    pb_mask = f_norm <= pb_edge
    ripple_db = spec_db[pb_mask].max() - spec_db[pb_mask].min()

    # Stopband attenuation
    sb_mask = f_norm >= sb_edge
    stopband_att_db = -spec_db[sb_mask].max()

    return ripple_db, stopband_att_db
ripple, stopband = analyze_chain_response(R, Fs_in, taps_q19, pb_edge=0.40, sb_edge=0.48)
print(f"Passband ripple ≈ {ripple:.3f} dB")



[SUCCESS] Stimulus written to cic_stimulus.txt, Expected outputs written to cic_expected.txt
[INFO] R=8, CIC_out=2048, FIR_taps=1
[INFO] Output range: -32768..32767
Passband ripple ≈ 2.130 dB
Stopband attenuation ≈ 0.0 dB
